# 03 · 從 Q 表到 DQN:用神經網路逼近 Q

上一課的 Q 表有個致命傷:**狀態一多就爆**。CartPole 的觀察是 4 個連續數值,組合無限多,根本列不成表。

**DQN(Deep Q-Network)** 的點子:既然列不成表,就用一個**神經網路**來逼近這個函式——輸入狀態,輸出每個動作的 Q 值。這一課我們**親手刻一個迷你 DQN**,把它在 CartPole 上練起來。

## 1. 安裝(torch 在 Colab 已內建)

In [ ]:
%pip install -q -U "gymnasium[classic-control]"

## 2. Q 網路 + 經驗回放

兩個 DQN 的關鍵工程:
- **Q 網路**:一個小 MLP,吃 4 維狀態、吐 2 個動作的 Q 值。
- **經驗回放 replay buffer**:把走過的 `(s,a,r,s',done)` 存起來,訓練時隨機抽一批,打散相關性、讓訓練穩定。

In [ ]:
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

device = "cuda" if torch.cuda.is_available() else "cpu"


class QNet(nn.Module):
    def __init__(self, n_obs, n_act):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_act),
        )

    def forward(self, x):
        return self.net(x)


buffer = deque(maxlen=10000)   # 經驗回放

## 3. 訓練迴圈

用 **target network**(一個慢半拍的 Q 網路複本)算目標值,避免「自己追自己」發散;epsilon 從 1 衰減到 0.05。

In [ ]:
env = gym.make("CartPole-v1")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

q = QNet(n_obs, n_act).to(device)
target = QNet(n_obs, n_act).to(device)
target.load_state_dict(q.state_dict())
opt = torch.optim.Adam(q.parameters(), lr=1e-3)

gamma, eps = 0.99, 1.0
batch_size = 64
rewards_hist = []

for ep in range(300):
    s, _ = env.reset()
    done, ep_reward = False, 0.0
    while not done:
        # epsilon-greedy 選動作
        if random.random() < eps:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                a = int(q(torch.tensor(s, dtype=torch.float32, device=device)).argmax())
        s2, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        buffer.append((s, a, r, s2, float(done)))
        s = s2; ep_reward += r

        # 從回放抽一批,做一次梯度更新
        if len(buffer) >= batch_size:
            batch = random.sample(buffer, batch_size)
            bs, ba, br, bs2, bd = map(np.array, zip(*batch))
            bs = torch.tensor(bs, dtype=torch.float32, device=device)
            bs2 = torch.tensor(bs2, dtype=torch.float32, device=device)
            ba = torch.tensor(ba, device=device).long()
            br = torch.tensor(br, dtype=torch.float32, device=device)
            bd = torch.tensor(bd, dtype=torch.float32, device=device)
            q_sa = q(bs).gather(1, ba.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                target_val = br + gamma * target(bs2).max(1).values * (1 - bd)
            loss = nn.functional.mse_loss(q_sa, target_val)
            opt.zero_grad(); loss.backward(); opt.step()

    eps = max(0.05, eps * 0.97)
    rewards_hist.append(ep_reward)
    if ep % 20 == 0:
        target.load_state_dict(q.state_dict())     # 定期同步 target network
        print(f"ep {ep:3d}  reward {ep_reward:5.0f}  eps {eps:.2f}")
print("訓練完成。")

## 4. 畫學習曲線

報酬一路往上爬,就代表 agent 真的學會平衡桿子了。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_rewards(rewards, window=50, title="Learning curve"):
    """畫每回合報酬與滑動平均，一眼看出 agent 有沒有在進步。"""
    plt.figure(figsize=(8, 4))
    plt.plot(rewards, alpha=0.3, label="episode reward")
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode="valid")
        plt.plot(range(window - 1, len(rewards)), ma, lw=2.2,
                 label=f"moving avg ({window})")
    plt.xlabel("episode"); plt.ylabel("reward"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
plot_rewards(rewards_hist, window=20, title="Hand-crafted DQN on CartPole")

## 小結

- **DQN = 用神經網路取代 Q 表**,解決連續/巨大狀態空間。
- 兩個讓訓練穩定的關鍵:**經驗回放** 與 **target network**。
- 親手刻一遍,你就懂了現成函式庫底下到底在做什麼。

下一課:同樣的事,用 **stable-baselines3** 一行搞定——並比較手刻與現成的差別。